<a href="https://colab.research.google.com/github/tinemyumi/saude-mental-datasus/blob/main/notebooks/3.tratamento_dados_sihsus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tratamento dos dados do SIH-SUS, IBGE e CNES**


**Conteúdo:**
Foram realizadas etapas de tratamento e preparação dos dados, incluindo verificação de valores nulos e duplicados, além de uma análise exploratória inicial para avaliação da consistência e qualidade da bases do SIH-SUS, IBGE, CNES  e dados referente aos Departamentos Regionais de Saúde (DRS).

**Autor:** Larissa Tinem


In [1]:
# Instale as bibliotecas necessárias
!pip install pandas
!pip install pyarrow

# **Conectando-se ao Google Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


# **Acessando dados no Google Drive**


In [3]:
# Código para acessar os dados no Google Drive

import pandas as pd
import os
import importlib
import sys

# Define o caminho da pasta onde os arquivos estão salvos no Google Drive
caminho_dados_drive = '/content/drive/MyDrive/Dados'

# Verifica se a pasta existe para evitar erros
if os.path.exists(caminho_dados_drive):
    print("A pasta foi encontrada no Google Drive. Listando arquivos...")

    # Lista todos os arquivos na pasta para visualização
    arquivos_na_pasta = os.listdir(caminho_dados_drive)
    print(arquivos_na_pasta)

else:
    print("Erro: A pasta de dados não foi encontrada. Verifique se o caminho está correto.")

print("\nProcesso concluído!")

A pasta foi encontrada no Google Drive. Listando arquivos...
['codigos_municipios_regioes.csv', 'populacao_municipios.csv', '.ipynb_checkpoints', 'Relatório de Progresso TCC 02.04.gdoc', 'cnes_estabelecimentos.csv', 'Leitos_2026.csv', 'dados_tratados', 'df_sihsus.parquet', 'df_sihsus_limpo.parquet', 'base_analitica']

Processo concluído!


# **1. Preparação e análise inicial do Dataset SIH-SUS**

## **1.1. Filtrando dados sobre saúde mental no Estado de São Paulo e transformação de dados**

- Os códigos CID-10 para transtornos mentais e comportamentais variam de F00 a F99.
- Código para o estado de São Paulo = 35
- Transformação de tipos de dados

In [4]:
import duckdb

df_sihsus = duckdb.query("""
SELECT
    N_AIH,
    UF_ZI,
    IDENT,
    ANO_CMPT,
    MES_CMPT,
    NASC,
    IDADE,
    SEXO,
    RACA_COR,
    ESPEC,
    MUNIC_RES,
    CEP,
    DIAG_PRINC,
    DIAG_SECUN,
    CAR_INT,
    COBRANCA,
    MUNIC_MOV,
    CID_ASSO,
    CID_MORTE,
    COMPLEX,
    DT_INTER,
    DT_SAIDA,
    QT_DIARIAS,
    DIAS_PERM,
    MORTE
FROM '/content/drive/MyDrive/Dados/df_sihsus.parquet'
WHERE DIAG_PRINC LIKE 'F%'
  AND UF_ZI LIKE '35%'
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
df_sihsus.head()

,N_AIH,UF_ZI,IDENT,ANO_CMPT,MES_CMPT,NASC,IDADE,SEXO,RACA_COR,ESPEC,...,COBRANCA,MUNIC_MOV,CID_ASSO,CID_MORTE,COMPLEX,DT_INTER,DT_SAIDA,QT_DIARIAS,DIAS_PERM,MORTE
0,3514126960003,351880,1,2015,01,19711030,43,3,01,05,...,12,351880,0000,0000,02,20141228,20141231,3,3,0
1,3514126960179,351880,1,2015,01,19720305,42,1,99,05,...,12,351880,0000,0000,02,20141216,20141220,4,4,0
2,3514126960180,351880,1,2015,01,19720925,42,3,99,05,...,12,351880,0000,0000,02,20141229,20150105,7,7,0
3,3514126960300,351880,1,2015,01,19760406,38,1,03,05,...,12,351880,0000,0000,02,20141230,20150106,7,7,0
4,3515101701397,351880,1,2015,01,19960220,18,3,03,05,...,12,351880,0000,0000,02,20150101,20150103,2,2,0


## **1.2. Transformação dos tipos de dados**

- Conversão das colunas **NASC, DT_INTER e DT_SAIDA** para formato data (aaaammdd)
- Conversão das colunas **ANO_CMPT, MES_CMPT, IDADE, QT_DIARIAS, DIAS_PERM** para números inteiros
- Renomar nome da coluna '**UF_ZI**' para cod_municipio para padronização com outras bases.
- Padronização da coluna 'cod_municipio' para 6 dígitos
- Classificar períodos da pandemia

In [6]:
import importlib
import pandas as pd

# Transformação dos tipos de dados

# Conversão para formato de data
df_sihsus['DT_INTER'] = pd.to_datetime(df_sihsus['DT_INTER'])
df_sihsus['DT_SAIDA'] = pd.to_datetime(df_sihsus['DT_SAIDA'])
df_sihsus['NASC'] = pd.to_datetime(df_sihsus['NASC'])

# Conversão para int
conversao_int = ['ANO_CMPT', 'MES_CMPT', 'IDADE', 'QT_DIARIAS', 'DIAS_PERM']
for coluna in conversao_int:
    df_sihsus[coluna] = df_sihsus[coluna].astype(int)


In [7]:
# Renomear a coluna UF_ZI para padronização com outros df
df_sihsus = df_sihsus.rename(columns={'UF_ZI': 'cod_municipio'})

# Padroniza cod_municipio para 6 dígitos
df_sihsus['cod_municipio'] = (
    df_sihsus['cod_municipio']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str[:6]
    .str.zfill(6)
)

In [8]:
# classificar periodos da pandemia
def classificar_periodo(row):
    ano = row['ANO_CMPT']
    mes = row['MES_CMPT']

    if (ano < 2020) or (ano == 2020 and mes <= 2):
        return 'pre-pandemia'

    elif (ano == 2020 and mes >= 3) or (ano in [2021]) or (ano == 2022 and mes <= 5):
        return 'pandemia'

    else:
        return 'pos-pandemia'

df_sihsus['periodo_pandemia'] = df_sihsus.apply(classificar_periodo, axis=1)

## **1.3. Salvando DF de saúde mental no Google Drive**

In [9]:
df_sihsus.to_parquet('/content/drive/MyDrive/Dados/dados_tratados/df_sihsus.parquet')

## **1.4. Análise Inicial**


In [10]:
# Informações sobre tipos de dado e quantidade de linhas e colunas
df_sihsus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1079670 entries, 0 to 1079669
Data columns (total 26 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   N_AIH             1079670 non-null  object        
 1   cod_municipio     1079670 non-null  object        
 2   IDENT             1079670 non-null  object        
 3   ANO_CMPT          1079670 non-null  int64         
 4   MES_CMPT          1079670 non-null  int64         
 5   NASC              1079670 non-null  datetime64[ns]
 6   IDADE             1079670 non-null  int64         
 7   SEXO              1079670 non-null  object        
 8   RACA_COR          1079670 non-null  object        
 9   ESPEC             1079670 non-null  object        
 10  MUNIC_RES         1079670 non-null  object        
 11  CEP               1079670 non-null  object        
 12  DIAG_PRINC        1079670 non-null  object        
 13  DIAG_SECUN        1079670 non-null  object

In [11]:
# Informações sobre valores ausentes
df_sihsus.isna().sum()


,0
N_AIH,0
cod_municipio,0
IDENT,0
ANO_CMPT,0
MES_CMPT,0
NASC,0
IDADE,0
SEXO,0
RACA_COR,0
ESPEC,0


In [12]:
# Informações sobre valores duplicados
duplicadas = df_sihsus.duplicated().sum()
print(duplicadas)

0


# **2. Preparação e análise inicial do dataset populacao_municipios**

In [13]:
import pandas as pd
df_ibge = pd.read_csv('/content/drive/MyDrive/Dados/populacao_municipios.csv')
df_ibge.head()

,ano,id_municipio,nome_municipio,sigla_uf,populacao
0,1970,1100106,Guajará-Mirim,RO,27016.0
1,1970,1100205,Porto Velho,RO,84048.0
2,1970,1200104,Brasiléia,AC,12311.0
3,1970,1200203,Cruzeiro do Sul,AC,43584.0
4,1970,1200302,Feijó,AC,15768.0


In [14]:
df_sp = df_ibge[df_ibge['sigla_uf'] == 'SP']

In [15]:
# separar anos
df_2022 = df_sp[df_sp['ano'] == 2022].copy()
df_2024 = df_sp[df_sp['ano'] == 2024].copy()

# merge por município
df_2023 = df_2022.merge(
    df_2024[['id_municipio', 'populacao']],
    on='id_municipio',
    suffixes=('_2022', '_2024')
)

# criar 2023 interpolado
df_2023['ano'] = 2023
df_2023['populacao'] = (
    df_2023['populacao_2022'] + df_2023['populacao_2024']
) / 2

# manter colunas necessárias
df_2023 = df_2023[['ano', 'id_municipio', 'nome_municipio', 'sigla_uf', 'populacao']]

# anexar ao df original
df_sp_final = pd.concat([df_sp, df_2023], ignore_index=True)

# ordenar
df_sp_final = df_sp_final.sort_values(['id_municipio', 'ano'])

In [16]:
print('Valores duplicados: ', df_sp_final.duplicated().sum())
print('Valores nulos: ')
print(df_sp_final.isna().sum())

Valores duplicados:  0
Valores nulos: 
ano               0
id_municipio      0
nome_municipio    0
sigla_uf          0
populacao         0
dtype: int64


In [17]:
# Renomear a coluna id_municipio para padronização com outros df
df_sp_final = df_sp_final.rename(columns={'id_municipio': 'cod_municipio'})

# Padroniza cod_municipio para 6 dígitos
df_sp_final['cod_municipio'] = (
    df_sp_final['cod_municipio']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str[:6]
    .str.zfill(6)
)

In [18]:
df_sp_final.to_csv('/content/drive/MyDrive/Dados/dados_tratados/df_populacao_sp', index=False)

# **3. Preparação e análise inicial CNES**

## **3.1. Estabelecimentos**

In [19]:
import pandas as pd

df_estabelecimentos = pd.read_csv('/content/drive/MyDrive/Dados/cnes_estabelecimentos.csv', sep=';', encoding='latin-1')
df_estabelecimentos.head(5)

/tmp/ipykernel_8579/2473984578.py:3: DtypeWarning: Columns (1,8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_estabelecimentos = pd.read_csv('/content/drive/MyDrive/Dados/cnes_estabelecimentos.csv', sep=';', encoding='latin-1')


,CO_CNES,CO_UNIDADE,CO_UF,CO_IBGE,NU_CNPJ_MANTENEDORA,NO_RAZAO_SOCIAL,NO_FANTASIA,CO_NATUREZA_ORGANIZACAO,DS_NATUREZA_ORGANIZACAO,TP_GESTAO,...,NO_EMAIL,CO_NATUREZA_JUR,ST_CENTRO_CIRURGICO,ST_CENTRO_OBSTETRICO,ST_CENTRO_NEONATAL,ST_ATEND_HOSPITALAR,ST_SERVICO_APOIO,ST_ATEND_AMBULATORIAL,CO_MOTIVO_DESAB,CO_AMBULATORIAL_SUS
0,19,2602900000019,26,260290,1.129440e+13,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA DR JAMACI DE MEDEIROS,NaN,NaN,M,...,NaN,1244.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,SIM
1,27,2602900000027,26,260290,1.093045e+13,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,CASA DE SAUDE SANTA HELENA,NaN,NaN,M,...,NaN,2062.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,SIM
2,35,2602900000035,26,260290,1.129440e+13,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,HOSPITAL MENDO SAMPAIO,NaN,NaN,M,...,NaN,1244.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,SIM
3,43,2602900000043,26,260290,1.129440e+13,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA DR MANUEL GOMES,NaN,NaN,M,...,NaN,1244.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,SIM
4,51,2602900000051,26,260290,1.129440e+13,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA VICENTE MENDES,NaN,NaN,M,...,NaN,1244.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,SIM


In [20]:
df_estabelecimentos.columns

Index(['CO_CNES', 'CO_UNIDADE', 'CO_UF', 'CO_IBGE', 'NU_CNPJ_MANTENEDORA',
       'NO_RAZAO_SOCIAL', 'NO_FANTASIA', 'CO_NATUREZA_ORGANIZACAO',
       'DS_NATUREZA_ORGANIZACAO', 'TP_GESTAO', 'CO_NIVEL_HIERARQUIA',
       'DS_NIVEL_HIERARQUIA', 'CO_ESFERA_ADMINISTRATIVA',
       'DS_ESFERA_ADMINISTRATIVA', 'CO_ATIVIDADE', 'TP_UNIDADE', 'CO_CEP',
       'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_BAIRRO', 'NU_TELEFONE',
       'NU_LATITUDE', 'NU_LONGITUDE', 'CO_TURNO_ATENDIMENTO',
       'DS_TURNO_ATENDIMENTO', 'NU_CNPJ', 'NO_EMAIL', 'CO_NATUREZA_JUR',
       'ST_CENTRO_CIRURGICO', 'ST_CENTRO_OBSTETRICO', 'ST_CENTRO_NEONATAL',
       'ST_ATEND_HOSPITALAR', 'ST_SERVICO_APOIO', 'ST_ATEND_AMBULATORIAL',
       'CO_MOTIVO_DESAB', 'CO_AMBULATORIAL_SUS'],
      dtype='object')

In [21]:
colunas_selecionadas = [
    'CO_CNES', 'CO_UNIDADE', 'CO_UF', 'CO_IBGE', 'NO_RAZAO_SOCIAL',
    'NO_FANTASIA', 'CO_ESFERA_ADMINISTRATIVA', 'CO_ATIVIDADE', 'TP_UNIDADE',
    'CO_CEP', 'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_BAIRRO', 'NU_LATITUDE', 'NU_LONGITUDE'
]
df_estabelecimentos_selecionado = df_estabelecimentos[colunas_selecionadas].copy()
df_estabelecimentos_selecionado.head()

,CO_CNES,CO_UNIDADE,CO_UF,CO_IBGE,NO_RAZAO_SOCIAL,NO_FANTASIA,CO_ESFERA_ADMINISTRATIVA,CO_ATIVIDADE,TP_UNIDADE,CO_CEP,NO_LOGRADOURO,NU_ENDERECO,NO_BAIRRO,NU_LATITUDE,NU_LONGITUDE
0,19,2602900000019,26,260290,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA DR JAMACI DE MEDEIROS,M,4,4,54580530,RUA 21 DE ABRIL,S/N,PONTE DOS CARVALHOS,-8.231660,-34.969599
1,27,2602900000027,26,260290,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,CASA DE SAUDE SANTA HELENA,M,4,5,54505560,AVN PRESIDENTE GETULIO VARGAS,428,CENTRO,-8.280193,-35.031437
2,35,2602900000035,26,260290,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,HOSPITAL MENDO SAMPAIO,M,4,5,54535430,BR 101 SUL KM 33,S/N,CHARNECA,-8.292214,-35.048198
3,43,2602900000043,26,260290,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA DR MANUEL GOMES,M,4,4,54510360,AV HISTORIADOR PEREIRA COSTA,S/N,CENTRO,-8.287650,-35.035000
4,51,2602900000051,26,260290,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,POLICLINICA VICENTE MENDES,M,4,4,54520250,RUA QUARENTA E HUM,S/N,COHAB,-8.295337,-35.032272


In [22]:
# ── 1. Filtrar SP ─────────────────────────────
df_caps_sp = df_estabelecimentos_selecionado[
    df_estabelecimentos_selecionado['CO_UF'].astype(str).str.startswith('35', na=False)
].copy()

# ── 2. Garantir tipo numérico ─────────────────
df_caps_sp['TP_UNIDADE'] = (
    df_caps_sp['TP_UNIDADE']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(float)
    .astype('Int64')
)

# ── 3. Filtro CAPS ────────────────────────────
CAPS_TP_UNIDADE = [70, 71, 72, 73, 74, 75, 76]

df_caps_sp = df_caps_sp[
    df_caps_sp['TP_UNIDADE'].isin(CAPS_TP_UNIDADE)
].copy()

# ── 4. Mapeamento ─────────────────────────────
mapa_caps = {
    70: 'CAPS_I',
    71: 'CAPS_II',
    72: 'CAPS_III',
    73: 'CAPS_i',
    74: 'CAPS_ad',
    75: 'CAPS_ad_III',
    76: 'CAPS_ad_IV'
}

df_caps_sp['tipo_caps'] = df_caps_sp['TP_UNIDADE'].map(mapa_caps)

print(f'Total de CAPS em SP: {len(df_caps_sp)}')
print(df_caps_sp['tipo_caps'].value_counts())

Total de CAPS em SP: 1760
tipo_caps
CAPS_I         667
CAPS_i         524
CAPS_ad        290
CAPS_ad_III    116
CAPS_II         80
CAPS_ad_IV      68
CAPS_III        15
Name: count, dtype: int64


In [23]:
# Renomear a coluna id_municipio para padronização com outros df
df_caps_sp = df_caps_sp.rename(columns={'CO_IBGE': 'cod_municipio'})

# Padroniza cod_municipio para 6 dígitos
df_caps_sp['cod_municipio'] = (
    df_caps_sp['cod_municipio']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str[:6]
    .str.zfill(6)
)

In [24]:
# Informações do df
df_caps_sp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1760 entries, 615 to 609558
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CO_CNES                   1760 non-null   int64  
 1   CO_UNIDADE                1760 non-null   object 
 2   CO_UF                     1760 non-null   int64  
 3   cod_municipio             1760 non-null   object 
 4   NO_RAZAO_SOCIAL           1760 non-null   object 
 5   NO_FANTASIA               1760 non-null   object 
 6   CO_ESFERA_ADMINISTRATIVA  1760 non-null   object 
 7   CO_ATIVIDADE              1760 non-null   int64  
 8   TP_UNIDADE                1760 non-null   Int64  
 9   CO_CEP                    1760 non-null   int64  
 10  NO_LOGRADOURO             1760 non-null   object 
 11  NU_ENDERECO               1760 non-null   object 
 12  NO_BAIRRO                 1760 non-null   object 
 13  NU_LATITUDE               1685 non-null   float64
 14  NU_LONGIT

In [25]:
# Checagem de valores nulos
df_caps_sp.isna().sum()

,0
CO_CNES,0
CO_UNIDADE,0
CO_UF,0
cod_municipio,0
NO_RAZAO_SOCIAL,0
NO_FANTASIA,0
CO_ESFERA_ADMINISTRATIVA,0
CO_ATIVIDADE,0
TP_UNIDADE,0
CO_CEP,0


In [26]:
# Checagem de valores duplicados
df_caps_sp.duplicated().sum()

np.int64(0)

In [27]:
# Salva o DataFrame filtrado como um arquivo CSV no Google Drive
df_caps_sp.to_csv('/content/drive/MyDrive/Dados/dados_tratados/df_estabelecimentos.csv', index=False)
print("DataFrame salvo com sucesso como 'df_estabelecimentos' no Google Drive.")

DataFrame salvo com sucesso como 'df_estabelecimentos' no Google Drive.


## **3.2. Leitos**

In [28]:
df_leitos = pd.read_csv('/content/drive/MyDrive/Dados/Leitos_2026.csv', sep=';', encoding='latin-1')
df_leitos.head(5)

,COMP,REGIAO,UF,CO_IBGE,MUNICIPIO,MOTIVO_DESABILITACAO,CNES,NOME_ESTABELECIMENTO,RAZAO_SOCIAL,TP_GESTAO,...,UTI_ADULTO_EXIST,UTI_ADULTO_SUS,UTI_PEDIATRICO_EXIST,UTI_PEDIATRICO_SUS,UTI_NEONATAL_EXIST,UTI_NEONATAL_SUS,UTI_QUEIMADO_EXIST,UTI_QUEIMADO_SUS,UTI_CORONARIANA_EXIST,UTI_CORONARIANA_SUS
0,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,NaN,27,CASA DE SAUDE SANTA HELENA,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,M,...,0,0,0,0,0,0,0,0,0,0
1,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,NaN,35,HOSPITAL MENDO SAMPAIO,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,...,0,0,0,0,0,0,0,0,0,0
2,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,NaN,94,MATERNIDADE PADRE GERALDO LEITE BASTOS,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,...,0,0,0,0,0,0,0,0,0,0
3,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,NaN,183,HOSPITAL SAMARITANO,SOCIEDADE HOSPITALAR SAMARITANO LTDA,M,...,0,0,0,0,0,0,0,0,0,0
4,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,NaN,221,HOSPITAL SAO SEBASTIAO,CASA DE SAUDE E MATERNIDADE SAO SEBASTIAO LTDA,M,...,10,0,0,0,0,0,0,0,0,0


In [29]:
df_leitos.columns

Index(['COMP', 'REGIAO', 'UF', 'CO_IBGE', 'MUNICIPIO', 'MOTIVO_DESABILITACAO',
       'CNES', 'NOME_ESTABELECIMENTO', 'RAZAO_SOCIAL', 'TP_GESTAO',
       'CO_TIPO_UNIDADE', 'DS_TIPO_UNIDADE', 'NATUREZA_JURIDICA',
       'DESC_NATUREZA_JURIDICA', 'NO_LOGRADOURO', 'NU_ENDERECO',
       'NO_COMPLEMENTO', 'NO_BAIRRO', 'CO_CEP', 'NU_TELEFONE', 'NO_EMAIL',
       'LEITOS_EXISTENTES', 'LEITOS_SUS', 'UTI_TOTAL_EXIST', 'UTI_TOTAL_SUS',
       'UTI_ADULTO_EXIST', 'UTI_ADULTO_SUS', 'UTI_PEDIATRICO_EXIST',
       'UTI_PEDIATRICO_SUS', 'UTI_NEONATAL_EXIST', 'UTI_NEONATAL_SUS',
       'UTI_QUEIMADO_EXIST', 'UTI_QUEIMADO_SUS', 'UTI_CORONARIANA_EXIST',
       'UTI_CORONARIANA_SUS'],
      dtype='object')

In [30]:
colunas_leitos_selecionadas = [
    'COMP', 'REGIAO', 'UF', 'CO_IBGE', 'MUNICIPIO', 'CNES', 'NOME_ESTABELECIMENTO',
    'RAZAO_SOCIAL', 'TP_GESTAO', 'CO_TIPO_UNIDADE', 'DS_TIPO_UNIDADE', 'NATUREZA_JURIDICA',
    'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_BAIRRO', 'CO_CEP',
    'LEITOS_EXISTENTES', 'LEITOS_SUS', 'UTI_TOTAL_EXIST', 'UTI_TOTAL_SUS',
    'UTI_ADULTO_EXIST', 'UTI_ADULTO_SUS', 'UTI_PEDIATRICO_EXIST', 'UTI_PEDIATRICO_SUS',
    'UTI_NEONATAL_EXIST', 'UTI_NEONATAL_SUS', 'UTI_QUEIMADO_EXIST', 'UTI_QUEIMADO_SUS',
    'UTI_CORONARIANA_EXIST', 'UTI_CORONARIANA_SUS'
]
df_leitos_selecionado = df_leitos[colunas_leitos_selecionadas].copy()
display(df_leitos_selecionado.head())

,COMP,REGIAO,UF,CO_IBGE,MUNICIPIO,CNES,NOME_ESTABELECIMENTO,RAZAO_SOCIAL,TP_GESTAO,CO_TIPO_UNIDADE,...,UTI_ADULTO_EXIST,UTI_ADULTO_SUS,UTI_PEDIATRICO_EXIST,UTI_PEDIATRICO_SUS,UTI_NEONATAL_EXIST,UTI_NEONATAL_SUS,UTI_QUEIMADO_EXIST,UTI_QUEIMADO_SUS,UTI_CORONARIANA_EXIST,UTI_CORONARIANA_SUS
0,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,27,CASA DE SAUDE SANTA HELENA,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,M,5,...,0,0,0,0,0,0,0,0,0,0
1,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,35,HOSPITAL MENDO SAMPAIO,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,5,...,0,0,0,0,0,0,0,0,0,0
2,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,94,MATERNIDADE PADRE GERALDO LEITE BASTOS,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,7,...,0,0,0,0,0,0,0,0,0,0
3,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,183,HOSPITAL SAMARITANO,SOCIEDADE HOSPITALAR SAMARITANO LTDA,M,5,...,0,0,0,0,0,0,0,0,0,0
4,202601,NORDESTE,PE,260290,CABO DE SANTO AGOSTINHO,221,HOSPITAL SAO SEBASTIAO,CASA DE SAUDE E MATERNIDADE SAO SEBASTIAO LTDA,M,5,...,10,0,0,0,0,0,0,0,0,0


In [31]:
# Filtra os leitos do estado de São Paulo
df_leitos_selecionado = df_leitos_selecionado[
    (df_leitos_selecionado['UF'] == 'SP') |
    (df_leitos_selecionado['CO_IBGE'].astype(str).str.startswith('35', na=False))
].copy()

# Exibe as primeiras linhas do DataFrame filtrado
display(df_leitos_selecionado.head())


,COMP,REGIAO,UF,CO_IBGE,MUNICIPIO,CNES,NOME_ESTABELECIMENTO,RAZAO_SOCIAL,TP_GESTAO,CO_TIPO_UNIDADE,...,UTI_ADULTO_EXIST,UTI_ADULTO_SUS,UTI_PEDIATRICO_EXIST,UTI_PEDIATRICO_SUS,UTI_NEONATAL_EXIST,UTI_NEONATAL_SUS,UTI_QUEIMADO_EXIST,UTI_QUEIMADO_SUS,UTI_CORONARIANA_EXIST,UTI_CORONARIANA_SUS
61,202601,SUDESTE,SP,353440,OSASCO,8028,HOSPITAL MUNICIPAL ANTONIO GIGLIO,PREFEITURA DO MUNICIPIO DE OSASCO,M,5,...,20,12,8,8,0,0,0,0,0,0
62,202601,SUDESTE,SP,353440,OSASCO,8036,HOSPITAL MATERNIDADE AMADOR AGUIAR,PREFEITURA DO MUNICIPIO DE OSASCO,M,7,...,0,0,0,0,18,18,0,0,0,0
63,202601,SUDESTE,SP,353440,OSASCO,8052,HOSPITAL REGIONAL DR VIVALDO MARTINS SIMOES OS...,SECRETARIA DE ESTADO DA SAUDE DE SAO PAULO,E,5,...,40,40,8,8,0,0,0,0,0,0
64,202601,SUDESTE,SP,353440,OSASCO,8087,PRONTO SOCORRO CONRADO CESARINO NUVOLINI,PREFEITURA DO MUNICIPIO DE OSASCO,M,20,...,0,0,0,0,0,0,0,0,0,0
65,202601,SUDESTE,SP,353440,OSASCO,8141,PRONTO SOCORRO DR ANTONIO FLAVIO FRANCA,PREFEITURA DO MUNICIPIO DE OSASCO,M,20,...,0,0,0,0,0,0,0,0,0,0


In [32]:
# Renomear a coluna id_municipio para padronização com outros df
df_leitos_selecionado = df_leitos_selecionado.rename(columns={'CO_IBGE': 'cod_municipio'})

# Padroniza cod_municipio para 6 dígitos
df_leitos_selecionado['cod_municipio'] = (
    df_leitos_selecionado['cod_municipio']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str[:6]
    .str.zfill(6)
)

In [33]:
# Verificação de valores nulos
df_leitos_selecionado.isna().sum()

,0
COMP,0
REGIAO,0
UF,0
cod_municipio,0
MUNICIPIO,0
CNES,0
NOME_ESTABELECIMENTO,0
RAZAO_SOCIAL,0
TP_GESTAO,0
CO_TIPO_UNIDADE,0


In [34]:
# Informações sobre o df
df_leitos_selecionado.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2049 entries, 61 to 14411
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   COMP                   2049 non-null   int64 
 1   REGIAO                 2049 non-null   object
 2   UF                     2049 non-null   object
 3   cod_municipio          2049 non-null   object
 4   MUNICIPIO              2049 non-null   object
 5   CNES                   2049 non-null   int64 
 6   NOME_ESTABELECIMENTO   2049 non-null   object
 7   RAZAO_SOCIAL           2049 non-null   object
 8   TP_GESTAO              2049 non-null   object
 9   CO_TIPO_UNIDADE        2049 non-null   int64 
 10  DS_TIPO_UNIDADE        2049 non-null   object
 11  NATUREZA_JURIDICA      2049 non-null   int64 
 12  NO_LOGRADOURO          2049 non-null   object
 13  NU_ENDERECO            2048 non-null   object
 14  NO_BAIRRO              2049 non-null   object
 15  CO_CEP                 2

In [35]:
# Verificação de valores duplicados
df_leitos_selecionado.duplicated().sum()

np.int64(0)

In [36]:
# Salva o DataFrame filtrado como um arquivo CSV no Google Drive
df_leitos_selecionado.to_csv('/content/drive/MyDrive/Dados/dados_tratados/df_leitos.csv', index=False)
print("DataFrame 'df_leitos_selecionado' salvo com sucesso como 'df_leitos.csv' no Google Drive.")

DataFrame 'df_leitos_selecionado' salvo com sucesso como 'df_leitos.csv' no Google Drive.


# **4. Preparação e análise inicial DRS**

In [37]:
df_drs = pd.read_csv('/content/drive/MyDrive/Dados/codigos_municipios_regioes.csv', sep=';', encoding='latin-1')
df_drs.head(5)

,cod_ibge,municipio,area_km,cod_ra,ra,cod_rm,rm,cod_drs,drs,cod_r_saude,r_saude
0,35,Estado de São Paulo,"248219,48",NaN,Estado de São Paulo,NaN,Estado de São Paulo,NaN,Estado de São Paulo,NaN,Estado de São Paulo
1,3500000,Sem especificação de município,NaN,NaN,Sem especificação de município,NaN,Sem especificação de município,NaN,Sem especificação de município,NaN,Sem especificação de município
2,3500105,Adamantina,"411,987",11.0,RA Presidente Prudente,9.0,Demais municípios,5.0,DRS Marília,35091.0,Adamantina
3,3500204,Adolfo,"211,055",9.0,RA São José do Rio Preto,9.0,RM São José do Rio Preto,15.0,DRS São José do Rio Preto,35156.0,José Bonifácio
4,3500303,Aguaí,"474,554",6.0,RA Campinas,9.0,Demais municípios,14.0,DRS São João da Boa Vista,35142.0,Mantiqueira


In [38]:
# Seleciona colunas desejadas no df
colunas_drs_selecionadas = [
    'cod_ibge', 'municipio', 'cod_drs', 'drs'
]
df_drs_selecionada = df_drs[colunas_drs_selecionadas].copy()

In [39]:
# Renomear a coluna id_municipio para padronização com outros df
df_drs_selecionada = df_drs_selecionada.rename(columns={'cod_ibge': 'cod_municipio'})

# Padroniza cod_municipio para 6 dígitos
df_drs_selecionada['cod_municipio'] = (
    df_drs_selecionada['cod_municipio']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str[:6]
    .str.zfill(6)
)

In [49]:
df_drs_selecionada[df_drs_selecionada.isna().any(axis=1)]

,cod_municipio,municipio,cod_drs,drs
0,000035,Estado de São Paulo,NaN,Estado de São Paulo
1,350000,Sem especificação de município,NaN,Sem especificação de município


In [53]:
df_drs_selecionada.duplicated().sum()

np.int64(0)

In [50]:
df_drs_selecionada.dropna()

,cod_municipio,municipio,cod_drs,drs
2,350010,Adamantina,5.0,DRS Marília
3,350020,Adolfo,15.0,DRS São José do Rio Preto
4,350030,Aguaí,14.0,DRS São João da Boa Vista
5,350040,Águas da Prata,14.0,DRS São João da Boa Vista
6,350050,Águas de Lindóia,3.0,DRS Campinas
...,...,...,...,...
642,355700,Votorantim,6.0,DRS Sorocaba
643,355710,Votuporanga,15.0,DRS São José do Rio Preto
644,355715,Zacarias,15.0,DRS São José do Rio Preto
645,355720,Chavantes,5.0,DRS Marília


In [51]:
# Salva o DataFrame filtrado como um arquivo CSV no Google Drive
df_drs_selecionada.to_csv('/content/drive/MyDrive/Dados/dados_tratados/df_drs.csv', index=False)
print("DataFrame 'df_drs_selecionada' salvo com sucesso como 'df_drs.csv' no Google Drive.")

DataFrame 'df_drs_selecionada' salvo com sucesso como 'df_drs.csv' no Google Drive.
